# A10 — Cờ kết quả nghiêm trọng theo triệu chứng

Heatmap cho biết trong các báo cáo có từng triệu chứng, có bao nhiêu báo cáo ghi cờ nhập viện, cấp cứu, đe dọa tính mạng, khuyết tật hoặc tử vong. Mỗi ô hiển thị phần trăm và số báo cáo có cờ / mẫu số.

Các cờ có thể chồng lấp. Trong core, giá trị không phải Y (kể cả giá trị thiếu) thành false, nên mẫu số có thể gồm báo cáo không ghi rõ outcome. Đây là phần trăm trong các báo cáo VAERS theo bộ lọc, không phải nguy cơ trên người được tiêm hay xác nhận nhân quả.

Heatmap title shows "all vaccines" when no vaccine filter is set; otherwise it displays the vaccine keyword used for a partial match.

## Bộ lọc tùy chọn

Nhập một phần tên sản phẩm trong `dim_vaccine.vax_name`, nhãn nhóm tuổi theo `dim_age_group`, và/hoặc khoảng `numdays` (ngày từ tiêm đến khởi phát). Bỏ trống các giá trị không muốn lọc. Để `selected_symptom = None` xem top N trong cohort sau lọc; nhập một symptom cụ thể để xem riêng symptom đó, kể cả khi không nằm trong top N.

## Thiết lập và kết nối

Cài dependency bằng `py -m pip install -r analysis/requirements.txt`. Notebook đọc PostgreSQL local bằng role analyst; cần `ANALYST_PASSWORD` trong `DVG6/.env`. Cần các relation core về report, vaccine, symptom, age group.

In [3]:
import sys
from pathlib import Path

project_dir = Path.cwd().resolve()
while not (project_dir / "Makefile").is_file() and project_dir != project_dir.parent:
    project_dir = project_dir.parent
if not (project_dir / "Makefile").is_file():
    raise FileNotFoundError("Cannot find the DVG6 project root")
sys.path.insert(0, str(project_dir))

from IPython.display import display
from analysis.notebook_utils import FIGURES_DIR, TABLES_DIR


In [9]:
from analysis.notebook_utils import query_df

# Xem các giá trị có thể dùng cho bộ lọc
display(query_df("SELECT DISTINCT dv.vax_name FROM core.dim_vaccine dv JOIN core.fct_report_vaccine rv ON rv.vaccine_key = dv.vaccine_key WHERE dv.vax_name <> 'UNKNOWN' ORDER BY 1"))
display(query_df("SELECT age_group_label FROM core.dim_age_group ORDER BY sort_order"))

vaccine_name = "PFIZER"  # None = all vaccines; or a keyword such as "PFIZER"
age_group = '19-30'     # ví dụ: '19-30'
onset_min_days = None
onset_max_days = None
selected_symptom = None  # None: top N; or enter a specific symptom
top_n = 10

from analysis.eda_workflows import run_a10
results = run_a10(
    vaccine_name=vaccine_name,
    age_group=age_group,
    onset_min_days=onset_min_days,
    onset_max_days=onset_max_days,
    selected_symptom=selected_symptom,
    top_n=top_n,
)
for result_name, result_frame in results.items():
    print(result_name)
    display(result_frame)

vax_name
str
"""COVID19 (COVID19 (JANSSEN))"""
"""COVID19 (COVID19 (MODERNA))"""
"""COVID19 (COVID19 (MODERNA MNEX…"
"""COVID19 (COVID19 (MODERNA UNKN…"
"""COVID19 (COVID19 (NOVAVAX))"""
"""COVID19 (COVID19 (PFIZER-BIONT…"
"""COVID19 (COVID19 (UNKNOWN))"""


age_group_label
str
"""5-11"""
"""12-15"""
"""16-18"""
"""19-30"""
"""31-50"""
"""51-65"""
"""66+"""
"""UNKNOWN"""


Percentages use distinct reports containing each symptom as denominator. Outcome flags may overlap; they are not mutually exclusive.
outcomes_by_symptom


symptom_name,report_denominator,hospital_reports,hospital_pct,er_reports,er_pct,life_threatening_reports,life_threatening_pct,disability_reports,disability_pct,death_reports,death_pct
str,i64,i64,"decimal[38,16]",i64,"decimal[38,16]",i64,"decimal[38,20]",i64,"decimal[38,20]",i64,"decimal[38,20]"
"""Headache""",3256,118,3.6240786240786241,489,15.0184275184275184,32,0.98280098280098280098,63,1.93488943488943490000,3,0.09213759213759213759
"""Dizziness""",2864,88,3.0726256983240223,711,24.8254189944134078,30,1.04748603351955310000,74,2.58379888268156420000,4,0.13966480446927374302
"""Fatigue""",2830,116,4.0989399293286219,490,17.3144876325088339,34,1.20141342756183750000,123,4.34628975265017670000,5,0.17667844522968197880
"""Pyrexia""",2783,135,4.8508803449514912,345,12.3966942148760331,30,1.07797340998922030000,39,1.40136543298598630000,4,0.14372978799856270212
"""Pain""",2624,144,5.4878048780487805,442,16.8445121951219512,28,1.06707317073170730000,105,4.00152439024390240000,3,0.11432926829268292683
"""Nausea""",2259,121,5.3563523683045595,447,19.7875166002656042,26,1.15095174856131030000,54,2.39043824701195220000,1,0.04426737494466578132
"""COVID-19""",2081,330,15.8577606919750120,187,8.9860643921191735,11,0.52859202306583373378,20,0.96107640557424315233,9,0.43248438250840941855
"""Chills""",2041,56,2.7437530622243998,233,11.4159725624693778,8,0.39196472317491425772,19,0.93091621754042136208,3,0.14698677119059284664
"""Chest pain""",1857,289,15.5627355950457728,842,45.3419493807215940,83,4.46957458266020460000,79,4.25417339795368870000,3,0.16155088852988691438
